<a href="https://colab.research.google.com/github/hsultova/Softuni-AI-Agents-Workflows/blob/main/langchain_memory_HITL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-openai langsmith

In [2]:
import os
import json
import operator

from pydantic import SecretStr
from typing import List, TypedDict, Annotated
from IPython.display import HTML, display

from google.colab import userdata
from langchain_openai import ChatOpenAI

from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.types import Command, Interrupt
from langchain.agents import create_agent, AgentState
from langchain.tools import ToolRuntime, tool

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

In [3]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_ENDPOINT"]="https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "Softuni-ai-agents"

openai_api_key = SecretStr(userdata.get("OPENAI_API_KEY"))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()


def print_interrupts(interrupts: List[Interrupt]):
    for interrupt in interrupts:
        for action_request in interrupt.value["action_requests"]:
            display(HTML(f'<div style="border: 1px dashed red; margin: 5px; padding: 10px; white-space: pre-wrap;">{action_request["description"]}</div>'))

In [4]:
openai_model = ChatOpenAI(model="gpt-4.1-mini", api_key=openai_api_key)

# Context and Short-Term Memory Setup

In [5]:
DESTINATIONS = {
    "santorini": {
        "name": "Santorini, Greece",
        "season": "Late spring to early autumn (May–September)",
        "from_euro": 350
    },
    "kyoto": {
        "name": "Kyoto, Japan",
        "season": "Spring (cherry blossoms) or Autumn (foliage)",
        "from_euro": 700
    },
    "bali": {
        "name": "Bali, Indonesia",
        "season": "Dry season (April–October)",
        "from_euro": 600
    },
    "lisbon": {
        "name": "Lisbon, Portugal",
        "season": "Year-round, best in spring/autumn",
        "from_euro": 150
    },
    "reykjavik": {
        "name": "Reykjavik, Iceland",
        "season": "Summer (June–August) for midnight sun, Winter for aurora",
        "from_euro": 300
    },
    "cape_town": {
        "name": "Cape Town, South Africa",
        "season": "Summer (November–March)",
        "from_euro": 550
    },
    "dubrovnik": {
        "name": "Dubrovnik, Croatia",
        "season": "Late spring to early autumn (May–September)",
        "from_euro": 200
    },
    "marrakech": {
        "name": "Marrakech, Morocco",
        "season": "Spring (March–May) or Autumn (September–November)",
        "from_euro": 180
    }
}

CATALOGUE = {
    "destinations": { key: dest["name"] for key, dest in DESTINATIONS.items() },
    "services": [
      "Chauffeured airport pickup with Blacklane or local equivalents",
      "Concierge-arranged restaurant reservations at Michelin-starred venues",
      "Personal travel curator for bespoke itinerary planning",
      "24/7 multilingual concierge support during the trip",
      "Private guided tours with vetted local historians",
      "Yacht charters for coastal or island-hopping segments",
      "Luxury villa or suite upgrades with dedicated butler service",
      "Spa and wellness packages pre-booked at partner resorts",
      "Fast-track airport security and lounge access",
      "Travel insurance with medical evacuation coverage",
      "On-call personal chef for private dining experiences",
      "Helicopter transfers for scenic or time-sensitive routes",
      "Custom photography sessions to document the trip",
      "Personal shopper for local designer or artisan goods"
    ]
}

CATALOGUE_AS_CONTEXT = json.dumps(CATALOGUE, indent=2)

In [6]:
class TravelConsultantContext(TypedDict):
  guest_id: str

class TravelConsultantState(AgentState):
  thread_notes: Annotated[list[str], operator.add]

#  Long-Term Memory Implementation
tools - remember_preference, recall_preferences

In [7]:
@tool
def remember_preference(key: str, value: str, runtime: ToolRuntime[TravelConsultantContext, TravelConsultantState]) -> str:
  """
  Store persistent guest preferences from the chat(e.g., dietary restrictions, seating preferences on flights, or favorite hotel brands).
  """
  namespace = ("guests", runtime.context["guest_id"], "preferences")
  runtime.store.put(namespace, key, {"value": value})
  return "User preferences stored successfully."

@tool
def recall_preferences(runtime: ToolRuntime[TravelConsultantContext]) -> str:
    """
    Read the full preference list for the guest. Call at the start of every session.
    """
    namespace = ("guests", runtime.context["guest_id"], "preferences")
    preferences = runtime.store.search(namespace, limit=100)
    result = []
    if preferences:
      result.append(
            "Guest preferences:\n" +
            "\n".join(f"- {i.key}: {i.value['value']}" for i in sorted(preferences, key=lambda i: i.key))
      )
    if not preferences:
      return "No prefernces stored. This is a new customer."

    return "\n\n".join(result)

In [8]:
SYSTEM_PROMPT = f"""You are an exclusive, high-end travel consultant. Help with the agency's premium services, available luxury destinations, and standard booking procedures.

# Catalogue you may sell from
{CATALOGUE_AS_CONTEXT}

# Operating rules

1. ALWAYS call `{recall_preferences.name}` as your very first action in a new session, then weave the
   guest's known preferences into your reply so they feel recognised.

2. Call `{remember_preference.name}` IMMEDIATELY — in the same turn, before your reply — any time the guest
   states a durable fact about themselves that isn't already in their recalled preferences. Do not wait
   until the end of the conversation, do not batch multiple mentions, and do not ask permission first.
   This is a background action; never tell the guest you're saving something, just do it.

   Durable preferences include (not exhaustive):
   - Dietary needs or allergies ("I'm vegetarian", "shellfish allergy")
   - Seating/cabin preferences ("I always fly aisle", "we prefer adjoining rooms")
   - Favourite brands, hotels, or airlines ("we love Aman properties")
   - Travel companions and their needs ("travelling with two kids, ages 6 and 9")
   - Activity interests ("we're avid divers", "not interested in nightlife")
   - Accessibility requirements
   - Anniversaries, celebrations, or recurring trip occasions

   Use stable, lowercase snake_case keys (e.g. `dietary`, `seat_preference`, `favourite_hotel_brand`,
   `travel_companions`, `accessibility`). If a new statement updates an existing key (e.g. preference
   changed), call `{remember_preference.name}` again with the same key and the new value — don't skip it
   just because something was recalled earlier under that key.

   Example: Guest says "My wife is celiac, so gluten-free options matter a lot to us." -> call
   `{remember_preference.name}` with key `dietary`, value describing the celiac/gluten-free requirement ->
   THEN reply, incorporating it naturally.

# Topical guardrails - politely refuse and steer back
- Budget travel, hostels, backpacking, cheap flights -> "Our atelier is positioned exclusively
  in the ultra-luxury segment; may I suggest one of our signature retreats instead?"
- Competing agencies (Abercrombie & Kent, Black Tomato, etc.) -> decline to compare; redirect.
- Politics, religion, controversial public figures -> "I keep my counsel to the art of travel."
- Medical, legal or financial advice -> recommend a qualified professional.
"""

In [9]:
checkpointer = InMemorySaver()
store = InMemoryStore()

In [10]:
travel_agent = create_agent(
    model=openai_model,
    system_prompt= SYSTEM_PROMPT,
    tools = [remember_preference, recall_preferences],
    middleware=[],
    context_schema=TravelConsultantContext,
    state_schema=TravelConsultantState,
    checkpointer=checkpointer,
    store=store)

In [11]:
guest1 = "Whitfield"
session1_config  = { "configurable": { "thread_id": f"{guest1}_2" } }
session1_context = {"guest_id": guest1}

session1_response = travel_agent.invoke(
    input={"messages": [HumanMessage("Hello! My name is Alexandra Whitfield and I'm planning a 7-night anniversary trip in mid-September for my husband and me. I'm vegetarian, I always need a window seat, and we love culture, fine dining, and relaxing at wellness resorts. Our budget is around €2500 per person, and it would be wonderful to arrange a private sunset dinner during the trip.")]},
    config=session1_config,
    context=session1_context)

In [12]:
print_conversation(session1_response["messages"])

================================ Human Message =================================

Hello! My name is Alexandra Whitfield and I'm planning a 7-night anniversary trip in mid-September for my husband and me. I'm vegetarian, I always need a window seat, and we love culture, fine dining, and relaxing at wellness resorts. Our budget is around €2500 per person, and it would be wonderful to arrange a private sunset dinner during the trip.
================================== Ai Message ==================================
Tool Calls:
  remember_preference (call_omnAgQElJQFhEAfElcjhReXg)
 Call ID: call_omnAgQElJQFhEAfElcjhReXg
  Args:
    key: name
    value: alexandra_whitfield
  remember_preference (call_yEkbC0dorvTxD3rxzqhQrtFL)
 Call ID: call_yEkbC0dorvTxD3rxzqhQrtFL
  Args:
    key: dietary
    value: vegetarian
  remember_preference (call_BadEAHDEXxoL3fqkvzMGY4DM)
 Call ID: call_BadEAHDEXxoL3fqkvzMGY4DM
  Args:
    key: seat_preference
    value: window
  remember_preference (call_Qh0FEKje9XGA

In [13]:
response2 = travel_agent.invoke(
    input={"messages": [HumanMessage("Do you know my name?")]},
    config=session1_config)

In [14]:
print_conversation(response2["messages"])

================================ Human Message =================================

Hello! My name is Alexandra Whitfield and I'm planning a 7-night anniversary trip in mid-September for my husband and me. I'm vegetarian, I always need a window seat, and we love culture, fine dining, and relaxing at wellness resorts. Our budget is around €2500 per person, and it would be wonderful to arrange a private sunset dinner during the trip.
================================== Ai Message ==================================
Tool Calls:
  remember_preference (call_omnAgQElJQFhEAfElcjhReXg)
 Call ID: call_omnAgQElJQFhEAfElcjhReXg
  Args:
    key: name
    value: alexandra_whitfield
  remember_preference (call_yEkbC0dorvTxD3rxzqhQrtFL)
 Call ID: call_yEkbC0dorvTxD3rxzqhQrtFL
  Args:
    key: dietary
    value: vegetarian
  remember_preference (call_BadEAHDEXxoL3fqkvzMGY4DM)
 Call ID: call_BadEAHDEXxoL3fqkvzMGY4DM
  Args:
    key: seat_preference
    value: window
  remember_preference (call_Qh0FEKje9XGA

In [15]:
session2_config  = { "configurable": { "thread_id": f"{guest1}_12" } }
session2_context = {"guest_id": guest1}

session2_response = travel_agent.invoke(
    input={"messages": [HumanMessage("I want to book a dinner in good restaurants for Sunday. What type of food you can recommend me?")]},
    config=session2_config,
    context=session2_context)

In [16]:
print_conversation(session2_response["messages"])

================================ Human Message =================================

I want to book a dinner in good restaurants for Sunday. What type of food you can recommend me?
================================== Ai Message ==================================
Tool Calls:
  recall_preferences (call_K3YHwbmxjMyxHZbuueHNwIhd)
 Call ID: call_K3YHwbmxjMyxHZbuueHNwIhd
  Args:
================================= Tool Message =================================
Name: recall_preferences

Guest preferences:
- dietary: vegetarian
- interests: culture, fine dining, wellness resorts
- name: alexandra_whitfield
- seat_preference: window
- travel_companions: husband
- trip_duration: 7 nights
- trip_occasion: anniversary
- trip_timing: mid september
================================== Ai Message ==================================

Alexandra, for a special Sunday dinner, given your vegetarian preference and love for fine dining and cultural experiences, I can recommend the following types of cuisine:

- Medi